# Capstone: Does Generation Order Matter?

An autoregressive model writes an image one pixel at a time. Before it can write
anything it needs an order, and the usual choice is raster: left to right, top to bottom,
the way you would read a page.

Nothing in the mathematics requires that. The chain rule factorizes a joint density
exactly the same way under any ordering of the variables:

$$p(x_1, \dots, x_D) = \prod_{i=1}^{D} p\big(x_{\pi(i)} \mid x_{\pi(1)}, \dots, x_{\pi(i-1)}\big)$$

for every permutation $\pi$. A model with unlimited capacity would fit every ordering to
the same held-out likelihood, because every ordering describes the same distribution.

Real models do not have unlimited capacity. That is the first question this project asks:
at a fixed parameter count, does the order change the fit, and if so, by enough to care
about? The second, if a different order does help, is whether the help came from the
structure you designed or simply from leaving raster behind.

Before you continue, make a prediction about the outcome that matters most:

<hr color="#A31F34">

<font color="#A31F34">Compared with raster order, how much will a different generation order change how well
the model fits held-out images?</font>

<hr color="#A31F34">

The results table will also show a supplied centre-out ordering alongside yours. This
gives you a useful reference point. If it does as well as your design, the gain came from
leaving raster rather than from the structure you chose.

Enter your prediction in the cell below before reading further. One word is sufficient.

In [ ]:
# Choose one:
#
#   "none"    no difference you could measure
#   "small"   a real difference, but too small to act on
#   "large"   big enough to change what you would build
#
# Not graded. The results cell near the end compares your prediction against what the
# run actually found.

FIRST_CALL = ""

> ### <font color="#A31F34">Big picture</font>
> <hr color="#A31F34">
> Every lab in this course handed you the intervention. You picked one of three prepared options and read the result. Here you design the intervention yourself: you author the generation order, you state what you expect it to do, and you declare in advance how large a difference would have to be before you would believe it.

## At a glance

| | |
|---|---|
| <font size="3">Runtime</font> | <font size="3">roughly 2 to 5 minutes on a free Colab GPU runtime, several times that on CPU</font> |
| <font size="3">Data</font> | <font size="3">FashionMNIST, downsampled to 14x14 and binarized, downloaded automatically</font> |
| <font size="3">Model</font> | <font size="3">a small MADE, 464,068 parameters, identical in every condition</font> |
| <font size="3">You design</font> | <font size="3">the generation order, the hypothesis, and the smallest difference worth reporting</font> |
| <font size="3">You submit</font> | <font size="3">seven values the last cell prints, copied to the course page</font> |

## What is graded

The last cell prints seven values and you copy them into the course page. Nothing asks
you to reproduce a number that moves from one run to the next.

| <font size="3">What it covers</font> | <font size="3">When you get it</font> |
|---|---|
| <font size="3">Three analysis helpers you write, run on inputs issued to you</font> | <font size="3">after Task 3</font> |
| <font size="3">Your proposal: that the design holds, and what it will cost</font> | <font size="3">before anything runs</font> |
| <font size="3">The record: that the run met its contract, and what it spent</font> | <font size="3">after the run</font> |

The probes have definite answers. The helpers are the same whichever path you took, so
switching paths keeps that work, but the inputs are issued per learner and yours differ
from your classmates'. Two further questions ask you to read a result and decide what to
run next; each asks you to select every statement that follows, so there may be more than
one.

Your comparison will land in one of three places, and all three are complete
findings. Reporting an inconclusive result honestly scores as well as reporting a
decisive one.

> ### <font color="#B45309">Watch out</font>
> <hr color="#B45309">
> Do not tune your generation order against the held-out set. Design it once, from a reason you can state, and report what it does. The point of declaring your hypothesis before you measure is that it stops you rewriting the prediction to match whatever came out.

## Setup

The first cell downloads the helper modules that run the checks. It works in Colab and
on a local machine, and skips the download if the files are already there.

In [ ]:
# Colab bootstrap: fetch the shared capstone modules if they are not already here.
import hashlib
import os
import urllib.request

REPO = ("https://raw.githubusercontent.com/codey-m/deep_learning/main/"
        "final_project/helpers")
NEEDED = ["project_schema.py", "made.py", "factorization_adapter.py"]
# Digests of the exact module versions this notebook was built and tested against. A
# file that does not match is a stale copy from an earlier session or a truncated
# download, and either one would fail later in a way that looks like your mistake.
DIGESTS = {
    "factorization_adapter.py": "95868f7e19a9a2d5a167ac86b9abe2147d32e5827e24fa9b609e2ccd60c48e09",
    "made.py": "eece233ce49fd07f2a070ab729df5acd7a670c63aae8f395869b13d7878ad077",
    "project_schema.py": "41d249621c4b4a0ab151d3355c32763a729b37cf90e760549b4688a95a826bd7",
}


def digest_of(path):
    with open(path, "rb") as handle:
        return hashlib.sha256(handle.read()).hexdigest()


for name in NEEDED:
    if not os.path.exists(name) or digest_of(name) != DIGESTS[name]:
        urllib.request.urlretrieve(f"{REPO}/{name}", name)
    if digest_of(name) != DIGESTS[name]:
        raise RuntimeError(
            f"{name} does not match the version this notebook was tested against. "
            f"Delete it and restart the runtime.")
print("helper modules ready:", ", ".join(NEEDED))

In [ ]:
import hashlib
import math
import statistics
import time

import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torchvision import datasets

import made
import project_schema as schema
import factorization_adapter as adapter
from project_schema import Contrast, ProjectPlan, RunRecord

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu")

SIDE = 14
DIMS = SIDE * SIDE
TRAIN_IMAGES, EVAL_IMAGES = 10000, 2000
HIDDEN, LAYERS = 512, 2
# QUICK_MODE is the graded default and is what the runtime estimate above assumes.
# Turning it off runs more seeds, which narrows every noise floor without changing
# the design.
QUICK_MODE = True
EPOCHS, BATCH = (20 if QUICK_MODE else 40), 256
SEEDS = tuple(7960 + i for i in range(10 if QUICK_MODE else 16))
MASK_SEED = 0
METRIC = "bits_per_dim"

print(f"device: {DEVICE}")
if DEVICE.type == "cpu":
    print("\nWARNING: no GPU is available, so this will run on CPU. The runtime above "
          "assumes a GPU and this will take several times longer. In Colab, use "
          "Runtime > Change runtime type > GPU, then run this cell again.")
print(f"instrument self-check: {made.self_check()}")

### The data

FashionMNIST at 28x28 is more pixels than this budget needs. Averaging each 2x2 block
down to 14x14 and thresholding at 0.5 gives 196 binary variables, which is small
enough to train thirty models in a few minutes and large enough that the ordering has
somewhere to matter.

The same 10,000 training images and 2,000 held-out images are used in every
condition. Bits per dimension is only comparable across conditions when the data
underneath it is identical, and the contract checks it.

In [ ]:
def load_binarized():
    """14x14 binarized FashionMNIST. Deterministic: no data sampling noise enters."""
    train = datasets.FashionMNIST("data", train=True, download=True)
    test = datasets.FashionMNIST("data", train=False, download=True)

    def prepare(dataset, count):
        images = dataset.data[:count].float() / 255.0
        images = nn.functional.avg_pool2d(images.unsqueeze(1), 2).squeeze(1)
        return (images > 0.5).float().reshape(len(images), -1)

    return prepare(train, TRAIN_IMAGES), prepare(test, EVAL_IMAGES)


train_data, eval_data = load_binarized()
train_data, eval_data = train_data.to(DEVICE), eval_data.to(DEVICE)
eval_digest = hashlib.sha256(
    eval_data.detach().cpu().numpy().tobytes()).hexdigest()[:16]
print(f"train {tuple(train_data.shape)}  held-out {tuple(eval_data.shape)}")
print(f"held-out digest {eval_digest}")

### The instrument, and the confound it is built to avoid

The model is a MADE: a feed-forward network whose weights are masked so that output
$i$ can only see inputs that come before $i$. Masking is what makes it autoregressive
without running $D$ separate networks.

Here is the trap. In a textbook MADE the masks are derived from the ordering, so
changing the order changes which connections survive. Two orderings then have
different numbers of live weights, and a comparison between them is measuring
capacity, not order. The finding would be real and the explanation would be wrong.

This implementation avoids it by separating the two ideas. The network is fixed and
always operates in rank space: position 0 of its input is whatever the ordering
says goes first. An ordering is applied by permuting pixels into rank space before
the model sees them, and permuting back afterwards. The masks never move.

> ### <font color="#1D4ED8">Intuition</font>
> <hr color="#1D4ED8">
> Same network, same masks, same live connections, every time. The only thing that changes between conditions is which pixel is called "first". That is what makes the comparison about order.

In [ ]:
# Two supplied orders. Read them: your own will be a third.
raster = made.raster_order(SIDE)          # reading order
centre_out = made.centre_out_order(SIDE)  # nearest the centre first, spiralling out

def show_orders(orders, title_map):
    figure, axes = plt.subplots(1, len(orders), figsize=(3.2 * len(orders), 3.4))
    for axis, (name, order) in zip(axes, orders.items()):
        rank = torch.empty(DIMS, dtype=torch.long)
        rank[order] = torch.arange(DIMS)
        axis.imshow(rank.reshape(SIDE, SIDE), cmap="viridis")
        axis.set_title(title_map[name], fontsize=10)
        axis.axis("off")
    figure.suptitle("when each pixel gets written (dark = early)", fontsize=11)
    plt.tight_layout()
    plt.show()

show_orders({"raster": raster, "centre_out": centre_out},
            {"raster": "raster", "centre_out": "centre-out"})

# The network is identical under both, which the contract will verify later.
signature = made.MADE(DIMS, HIDDEN, LAYERS, mask_seed=MASK_SEED).connectivity_signature()
print(f"connectivity signature: {signature}")

### Why raster might lose

Raster order writes the whole top of the image before it writes anything at the
bottom. By the time the model predicts a pixel in the last row, everything it is
conditioning on is at least one full row away, and often much further. Centre-out
scatters the early pixels instead, so later pixels tend to have a written neighbour
nearby.

That is an argument, not a result. The model has 464,068 parameters and might have
enough capacity that neither ordering strains it. Measuring is the point.

### Before Task 1: your issued inputs

Checkpoint 1 on the course page issues you a parameter count, a step count, a batch
size, and a three-seed table of (baseline, treatment) pairs. They are drawn per learner,
so yours differ from your classmates' and the three values you submit have to come from
your own helpers.

Copy them into the block below. You can leave the zeros for now: every probe below runs
either way, and the fixed probes are what tell you your helpers work.

In [ ]:
# From Checkpoint 1 on the course page. Replace the zeros with your issued values.
ISSUED_PARAMETERS = 0
ISSUED_STEPS = 0
ISSUED_BATCH = 0
ISSUED_TABLE = {
    1: (0.0, 0.0),
    2: (0.0, 0.0),
    3: (0.0, 0.0),
}

issued_ready = ISSUED_PARAMETERS > 0 and ISSUED_TABLE[1] != (0.0, 0.0)
print("issued inputs:", "loaded" if issued_ready else "not filled in yet")

## Your task 1: Scope a run to a compute budget

Every lab so far handed you a compute budget. Estimating one is a skill in itself,
and it is the difference between an experiment that finishes inside a Colab session
and one that dies partway through with nothing to show.

Use the cheapest estimate that tracks real cost: the parameters that do work on
each example, multiplied by the number of examples pushed through. Here that is the whole model, because
every parameter of it is trained. Return the count in billions, so the numbers stay readable:

$$\text{budget units} = \frac{\text{trainable parameters} \times \text{steps} \times \text{batch size}}{10^9}$$

> ### <font color="#1D4ED8">Intuition</font>
> <hr color="#1D4ED8">
> Budget units will not predict wall-clock seconds on a particular GPU. They are for ranking designs against each other and for catching the one that is a hundred times larger than you thought.

In [ ]:
# STUDENT TASK 1: estimate the cost of a run in budget units.
def compute_budget(trainable_parameters, steps, batch_size):
    """Trainable parameters times examples processed, in billions."""
    # TODO: return the cost in budget units.
    return 0.0

In [ ]:
# Probe 1 (fixed input, definite answer): 35,000 parameters, 250 steps, batch size 50.
# This one never changes, so it tells you whether compute_budget works at all.
probe_budget_value = round(compute_budget(35_000, 250, 50), 4)
budget_probe_contract = abs(probe_budget_value - 0.4375) < 1e-4
print(f"R1 probe: {probe_budget_value} budget units "
      f"({'matches' if budget_probe_contract else 'does not match'} the expected 0.4375)")

# The value you submit comes from your own issued inputs.
if issued_ready:
    issued_budget_value = round(
        compute_budget(ISSUED_PARAMETERS, ISSUED_STEPS, ISSUED_BATCH), 4)
    print(f"R1 to submit: {issued_budget_value} budget units")
else:
    issued_budget_value = None
    print("R1 to submit: fill in the issued block above first")

## Your task 2: Compare two conditions the paired way

You will train each ordering under ten random seeds. Seeds change the weight
initialization and the batch order, so the same ordering does not give the same
number twice.

The naive comparison takes the mean of one condition and subtracts the mean of the
other. The paired comparison takes the difference within each seed first, then
averages those differences.

With the same seeds in both conditions these two give the identical mean. What
changes is the uncertainty around it. Seed 7963 might be a poor initialization for
every ordering at once, and when you difference within that seed, its badness
cancels. Differencing the group means instead leaves that easiness in the comparison.

How much pairing buys depends on how much the two conditions actually share, so it
is not a fixed number and is not worth taking on faith. When you read the result you
will see both floors, the paired one and the one you would have got by differencing
the group means, and you can judge the size of the difference on your own run.

In [ ]:
# STUDENT TASK 2: the mean within-seed difference between two conditions.
def paired_difference(treatment_by_seed, reference_by_seed):
    """Both arguments map seed -> metric value. Use only the seeds present in both."""
    seeds = sorted(set(treatment_by_seed) & set(reference_by_seed))
    # TODO: build the list of within-seed differences (treatment minus reference),
    # then return their mean.
    return 0.0

In [ ]:
# Probe 2 (fixed input, definite answer): three seeds, two conditions.
PROBE_TREATMENT = {1: 0.30, 2: 0.28, 3: 0.26}
PROBE_REFERENCE = {1: 0.34, 2: 0.33, 3: 0.29}
probe_paired_value = round(paired_difference(PROBE_TREATMENT, PROBE_REFERENCE), 4)
paired_probe_contract = abs(probe_paired_value - (-0.04)) < 1e-4
print(f"R2 probe: {probe_paired_value} "
      f"({'matches' if paired_probe_contract else 'does not match'} the expected -0.04)")

if issued_ready:
    issued_treatment = {seed: ISSUED_TABLE[seed][1] for seed in (1, 2, 3)}
    issued_reference = {seed: ISSUED_TABLE[seed][0] for seed in (1, 2, 3)}
    issued_paired_value = round(
        paired_difference(issued_treatment, issued_reference), 4)
    print(f"R2 to submit: {issued_paired_value}")
else:
    issued_paired_value = None
    print("R2 to submit: fill in the issued block above first")

## Your task 3: Decide when a difference is big enough to believe

A mean difference on its own settles nothing. Ten seeds is a small sample, and a
difference of 0.01 means one thing when the seed-to-seed spread is 0.001 and another
thing entirely when the spread is 0.05.

The floor is the half-width of a t-interval on the paired differences:

$$\text{floor} = t_{0.975,\, n-1} \cdot \frac{s}{\sqrt{n}}$$

where $s$ is the sample standard deviation of the within-seed differences and $n$ is
how many you have. A difference is resolved when its magnitude exceeds its own
floor. Anything smaller is inside the noise your own seeds produce, and you cannot
tell it apart from zero.

The critical value is supplied. You write the arithmetic that combines it with the
spread and the count.

> ### <font color="#B45309">Watch out</font>
> <hr color="#B45309">
> Resolving a difference and caring about one are separate questions. Your SESOI, the smallest effect size of interest, answers the second: the smallest change you would act on, declared before you measure. The verdict below combines both. If the whole interval sits beyond your SESOI, the effect is at least the size you said you would act on. If the whole interval sits inside it, the effect is at most that size, which is a real result rather than a failure. If the interval straddles it, this many seeds cannot separate the two.

In [ ]:
# STUDENT TASK 3: the paired noise floor for a list of within-seed differences.
def resolution_floor(differences, critical_value):
    """Half-width of the t-interval around the mean of ``differences``."""
    count = len(differences)
    # TODO: return the half-width of the t-interval around their mean.
    return 0.0

In [ ]:
# Probe 3 (fixed input, definite answer): the same three seeds as probe 2.
PROBE_DIFFERENCES = [PROBE_TREATMENT[s] - PROBE_REFERENCE[s] for s in (1, 2, 3)]
probe_floor_value = round(
    resolution_floor(PROBE_DIFFERENCES, schema._critical(len(PROBE_DIFFERENCES))), 4)
floor_probe_contract = abs(probe_floor_value - 0.0248) < 1e-4
print(f"R3 probe: {probe_floor_value} "
      f"({'matches' if floor_probe_contract else 'does not match'} the expected 0.0248)")
print(f"the probe difference of {probe_paired_value} is "
      f"{'resolved' if abs(probe_paired_value) > probe_floor_value else 'inside the noise'}")

if issued_ready:
    issued_differences = [ISSUED_TABLE[seed][1] - ISSUED_TABLE[seed][0]
                          for seed in (1, 2, 3)]
    issued_floor_value = round(
        resolution_floor(issued_differences,
                         schema._critical(len(issued_differences))), 4)
    print(f"R3 to submit: {issued_floor_value}")
    print(f"your issued difference of {issued_paired_value} is "
          f"{'resolved' if abs(issued_paired_value) > issued_floor_value else 'inside the noise'}")
else:
    issued_floor_value = None
    print("R3 to submit: fill in the issued block above first")

## Your task 4: Design your generation order

This is the decision the project asks you to author. Write a function returning a permutation of
$0, \dots, 195$: entry $i$ is the pixel index written at rank $i$.

The only hard requirements are that it is a genuine permutation, so every pixel
appears exactly once, and that it is not one of the two supplied orders. Beyond that
the decision is yours, and it should follow from a reason you can state in one
sentence.

<details style="border:1px solid #e5e7eb;border-radius:8px;padding:10px 14px;background:#f9fafb;color:#111827;margin:14px 0;">
<summary style="cursor:pointer;font-weight:600;">Families of orders worth considering</summary>
<div style="margin-top:8px;"><ul><li><b>Coarse to fine.</b> Write every fourth pixel, then every second, then the rest. Early pixels form a low-resolution sketch of the whole image.</li><li><b>Checkerboard.</b> All the black squares, then all the white ones. Every late pixel has four written neighbours.</li><li><b>Random.</b> A permutation with no structure at all. Useful as a reference point: it tells you whether structure is doing anything or whether merely leaving raster order is enough.</li><li><b>Space-filling curve.</b> A boustrophedon or Hilbert traversal keeps consecutive ranks physically adjacent, which is the opposite bet to coarse-to-fine.</li><li><b>Content driven.</b> Order by how often a pixel is on across the training set. Note that this reads the training data, which is allowed, but reading the held-out data is not.</li></ul></div>
</details>

In [ ]:
# STUDENT TASK 4: author your generation order.
def learner_order(side):
    """Return a LongTensor permutation of 0..side*side-1: entry i is written at rank i."""
    # TODO: replace this with your own order. It must be a permutation, and it must
    # differ from both raster_order and centre_out_order.
    return made.raster_order(side)


MY_ORDER_REASON = ""  # TODO: one sentence, at least 6 words, on why you expect
                      # this order to behave as it does.

In [ ]:
# Timed for the same reason the other paths time their authored code: R5 and R7
# price the supplied model, not whatever your ordering function does to produce a
# permutation. Diagnostic only, and not comparable between machines.
_order_started = time.perf_counter()
learner = learner_order(SIDE)
order_seconds = time.perf_counter() - _order_started
orders = {"raster": raster, "centre_out": centre_out, "learner": learner}

order_is_permutation = made.is_permutation(learner, DIMS)
order_is_novel = not any(torch.equal(learner, supplied)
                         for supplied in (raster, centre_out))
print(f"permutation: {order_is_permutation}   differs from both supplied orders: "
      f"{order_is_novel}")

show_orders(orders, {"raster": "raster", "centre_out": "centre-out",
                     "learner": "yours"})

## Your task 5: Declare your prediction before you measure

Four declarations, all made before a single model is trained.

**The hypothesis.** What you expect your order to do to held-out bits per dimension,
and the mechanism you think would cause it. Bits per dimension is a loss, so lower is
better.

**The required contrast.** Which comparison your project stands on: your order
against raster, or your order against centre-out. Also its direction. Predicting that
your order does worse is a legitimate prediction, and if you believe it, declare it.

**The smallest effect worth caring about.** Expressed as a fraction of the raster
baseline. Declaring 0.02 says a 2 percent improvement in bits per dimension would
change what you would build; anything smaller would not, even if you could resolve
it. This is the number that lets a null result be a finding rather than a failure.

**The budget.** What the whole experiment will cost, from Task 1.

> ### <font color="#A31F34">Big picture</font>
> <hr color="#A31F34">
> Declaring the effect size you care about before measuring is what separates an experiment from a search. If you fix it afterwards, you will fix it wherever your result happened to land, and you will have learned nothing you did not already assume.

In [ ]:
# STUDENT TASK 5: declare the design.
MY_HYPOTHESIS = ""  # TODO: at least 12 words. What you expect, and the
                    # mechanism you think causes it.

# TODO: which comparison does your project stand on? treatment is always "learner";
# reference is "raster" or "centre_out"; direction is "less" (your order improves the
# loss) or "greater" (your order worsens it).
REQUIRED_CONTRAST = Contrast("learner_vs_raster", "learner", "raster",
                             required=True, direction="less")

# TODO: the smallest relative change in bits per dimension you would act on.
# Must be above 0 and at most 0.25.
SESOI_RELATIVE = 0.02

In [ ]:
CONDITIONS = ("raster", "centre_out", "learner")
CONTRASTS = (
    REQUIRED_CONTRAST,
    Contrast("centre_vs_raster", "centre_out", "raster", direction="less"),
    Contrast("learner_vs_centre", "learner", "centre_out", direction="less"),
)
configs = {c: {"generation_order": c, "mask_seed": MASK_SEED} for c in CONDITIONS}

parameters = sum(p.numel() for p in made.MADE(DIMS, HIDDEN, LAYERS).parameters())
steps_per_run = EPOCHS * math.ceil(TRAIN_IMAGES / BATCH)
projected_budget_units = round(
    compute_budget(parameters, steps_per_run, BATCH) * len(CONDITIONS) * len(SEEDS), 4)

plan = ProjectPlan(
    path="factorization_order",
    question="Under fixed capacity, does the order in which pixels are generated "
             "change how well the model fits held-out images?",
    hypothesis=MY_HYPOTHESIS,
    control="raster",
    intervention="generation_order",
    declared_change="generation_order",
    conditions=CONDITIONS,
    evaluation_slices=(adapter.TRAIN_SLICE, adapter.EVAL_SLICE),
    seeds=SEEDS,
    compute_budget=projected_budget_units,
    decisions=(f"mask_seed={MASK_SEED}", f"dims={DIMS}",
               f"sesoi={SESOI_RELATIVE}"),
    condition_kind="categorical",
    contrasts=CONTRASTS,
    # One fixed dataset for every seed. Replication is over initialization and batch
    # order, not over which images were drawn.
    seed_varies="training_only",
)

BUDGET_MIN, BUDGET_MAX = 0.0, 20000.0

# The generic proposal contract, run now rather than after the experiment. This is the
# same check the execution contract applies to the plan, so a malformed design fails here
# in a second instead of after the run.
proposal = schema.ContractResult()
schema.check_plan(plan, budget_min=BUDGET_MIN, budget_max=BUDGET_MAX, result=proposal)

design_contract, design_report = schema.checklist({
    "your order is a genuine permutation of every pixel": order_is_permutation,
    "it differs from both supplied orders": order_is_novel,
    "MY_HYPOTHESIS is at least 12 words": len(MY_HYPOTHESIS.split()) >= 12,
    "MY_ORDER_REASON is at least 6 words": len(MY_ORDER_REASON.split()) >= 6,
    "your required contrast is marked required": REQUIRED_CONTRAST.required,
    "its treatment is 'learner'": REQUIRED_CONTRAST.treatment == "learner",
    "its reference is 'raster' or 'centre_out'":
        REQUIRED_CONTRAST.reference in ("raster", "centre_out"),
    "its direction is 'less' or 'greater'":
        REQUIRED_CONTRAST.direction in ("less", "greater"),
    "SESOI_RELATIVE is above 0 and at most 0.25": 0.0 < SESOI_RELATIVE <= 0.25,
    "your projected budget is above zero (Task 1 is finished)":
        projected_budget_units > 0,
    "the plan passes the generic proposal checks printed below": bool(proposal.passed),
})

print(f"model parameters: {parameters:,}   steps per run: {steps_per_run}")
print(f"R4 design contract: {design_contract}")
print(design_report)
if not proposal.passed:
    print(proposal.report())
print(f"R5 projected budget: {projected_budget_units} units for "
      f"{len(CONDITIONS)} orders x {len(SEEDS)} seeds")
print(f"plan digest: {plan.freeze()}")

# Bind every measurement to the design and to the code that produced it. The bytecode
# and constants of your own function go into the digest alongside the frozen plan, so a
# result table can be traced to the exact method that made it and a plan edited after
# the run stops matching its own rows. A notebook can always be re-run from the top,
# so this detects a changed design rather than preventing one.
provenance = hashlib.sha256(
    plan.freeze().encode("utf-8")
    + learner_order.__code__.co_code
    + repr(learner_order.__code__.co_consts).encode("utf-8")).hexdigest()[:16]
print(f"run provenance: {provenance}")

> ### <font color="#B45309">Watch out</font>
> <hr color="#B45309">
> R4 must read 1 before you go on. If it does not, one of the declarations above is still a placeholder, or your order is not a permutation. Running the experiment on a design that fails its own contract wastes the compute and the result will not be gradeable.

## Run the experiment

The training loop is supplied, because the training loop is not what this project is
about. Read it once anyway. Two lines carry the whole design.

The first is `train_data[:, order]`, which permutes the pixels into rank space. That
single indexing operation is the entire intervention.

The second is `MADE(DIMS, HIDDEN, LAYERS, mask_seed=MASK_SEED)`, constructed with the
same mask seed in every condition. A different mask seed would be a different
network, and the contract rejects it.

In [ ]:
def train_one(order, seed):
    """Train the fixed network on pixels permuted into the given rank order."""
    torch.manual_seed(seed)
    model = made.MADE(DIMS, HIDDEN, LAYERS, mask_seed=MASK_SEED).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    ranked = train_data[:, order.to(DEVICE)]
    shuffler = torch.Generator().manual_seed(seed * 31)
    examples_processed = 0
    for _ in range(EPOCHS):
        permutation = torch.randperm(len(ranked), generator=shuffler).to(DEVICE)
        for start in range(0, len(ranked), BATCH):
            batch = ranked[permutation[start:start + BATCH]]
            loss = nn.functional.binary_cross_entropy_with_logits(model(batch), batch)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            examples_processed += len(batch)
    return model, examples_processed


started = time.time()
records, signature_samples, hash_samples = [], {}, {}
measured_budget_units, last_model = 0.0, None
split_id = schema.split_hash(list(range(TRAIN_IMAGES)), list(range(EVAL_IMAGES)))

for seed in SEEDS:
    for condition in CONDITIONS:
        order = orders[condition].to(DEVICE)
        model, examples_processed = train_one(orders[condition], seed)
        # Collected per seed, so the connectivity check compares every seed's models.
        signature_samples.setdefault(condition, []).append(
            str(model.connectivity_signature()))
        hash_samples.setdefault(condition, []).append(eval_digest)
        last_model = model
        # The projection above assumed every batch was full. This counts the examples
        # that actually went through, and the last batch of each epoch is short, so
        # the two numbers are close without being the same number twice.
        measured_budget_units += compute_budget(parameters, examples_processed, 1)
        for slice_name, data in ((adapter.TRAIN_SLICE, train_data),
                                 (adapter.EVAL_SLICE, eval_data)):
            records.append(RunRecord(
                condition=condition, seed=seed, split_hash=split_id,
                config_hash=schema.config_hash(configs[condition]),
                training_steps=EPOCHS, runtime_seconds=order_seconds,
                provenance=provenance,
                metric_name=METRIC, evaluation_slice=slice_name,
                value=made.bits_per_dim(model, data, order)))
    print(f"  seed {seed} done ({time.time() - started:.0f}s elapsed)", flush=True)

# Tuples, so "identical across every order" means identical on every seed too.
signatures = {c: tuple(v) for c, v in signature_samples.items()}
eval_hashes = {c: tuple(v) for c, v in hash_samples.items()}

measured_budget_units = round(measured_budget_units, 4)
print(f"\n{len(records)} records in {time.time() - started:.0f}s")
print(f"budget projected {projected_budget_units}, measured {measured_budget_units}")

## Read the result

Three numbers per ordering: the mean held-out bits per dimension, the spread across
seeds, and the paired comparison against the reference you declared.

Look at the paired column, not the means column. Two means that look far apart can
sit inside a noise floor that is wider still.

In [ ]:
values = schema.paired_values(records, METRIC, adapter.EVAL_SLICE)
means = {c: statistics.mean(values[c].values()) for c in CONDITIONS}
# Your SESOI was declared as a fraction of the baseline, so it becomes an absolute
# number only once the baseline is measured. The verdicts are judged against it.
sesoi_absolute = SESOI_RELATIVE * means["raster"]
report = schema.contrast_report(records, METRIC, adapter.EVAL_SLICE, CONTRASTS,
                                sesoi=sesoi_absolute)

print(f"{'order':<12} {'bits/dim':>10} {'sd':>8} {'vs raster':>11}")
for condition in CONDITIONS:
    column = [values[condition][s] for s in SEEDS]
    delta = means[condition] - means["raster"]
    change = "" if condition == "raster" else f"{delta / means['raster']:+.1%}"
    print(f"{condition:<12} {means[condition]:>10.4f} "
          f"{statistics.stdev(column):>8.4f} {change:>11}")

print()
print(schema.format_contrasts(report, label="held-out bits/dim"))
print(f"\n(your SESOI of {SESOI_RELATIVE:.0%} is {sesoi_absolute:.4f} bits/dim at "
      f"this baseline)")

# Your own helpers, applied to your own experiment. They must agree with the report.
required_row = next(row for row in report["rows"]
                    if row["name"] == REQUIRED_CONTRAST.name)
own_differences = [values[REQUIRED_CONTRAST.treatment][s]
                   - values[REQUIRED_CONTRAST.reference][s] for s in SEEDS]
own_mean = paired_difference(values[REQUIRED_CONTRAST.treatment],
                             values[REQUIRED_CONTRAST.reference])
own_floor = resolution_floor(own_differences, schema._critical(len(own_differences)))
helpers_agree = (abs(own_mean - required_row["mean"]) < 1e-9
                 and abs(own_floor - required_row["floor"]) < 1e-9)
print(f"\nyour helpers reproduce the report: {helpers_agree}")

# What pairing bought on this run: the same contrast judged by differencing the group
# means instead of within each seed. The factor depends on how much the two conditions
# share, so it is measured here rather than asserted.
_treat = [values[REQUIRED_CONTRAST.treatment][s] for s in SEEDS]
_ref = [values[REQUIRED_CONTRAST.reference][s] for s in SEEDS]
unpaired_floor = schema._critical(len(SEEDS)) * math.sqrt(
    (statistics.stdev(_treat) ** 2 + statistics.stdev(_ref) ** 2) / len(SEEDS))
if own_floor > 0:
    print(f"floor on your required contrast: {unpaired_floor:.4f} unpaired against "
          f"{own_floor:.4f} paired ({unpaired_floor / own_floor:.1f}x)")
    print("  pairing only buys something where the two conditions rise and fall "
          "together across seeds. A factor near 1.0 means yours barely do, which is a "
          "fact about the conditions rather than a mistake, and is worth a sentence in "
          "your caveat.")
else:
    print("floor comparison skipped: your resolution_floor returned 0, so Task 3 is "
          "not finished. Everything below this point depends on it.")

# The call you made at the top, before the notebook had shown you anything, set
# against what the run found. It is judged against your required contrast, which
# is the comparison the opening question asks about unless you changed the
# reference in Task 5. If you did, read this line against the contrast you chose.
first_call = str(globals().get("FIRST_CALL", "")).strip().lower()
if first_call in ("none", "small", "large"):
    called_large = first_call == "large"
    verdict = required_row["verdict"]
    if verdict == "inconclusive":
        outcome = ("this run cannot separate those two possibilities, so your call "
                   "stands untested rather than wrong")
    elif (verdict == "meaningful") == called_large:
        outcome = "the run agrees with you"
    else:
        outcome = ("the run disagrees with you, which is the more interesting of the "
                   "two outcomes and belongs in your record")
    print(f"\nyour first call was {first_call!r} and the effect came out {verdict}: "
          f"{outcome}.")
else:
    print("\nno first call was recorded at the top of the notebook")

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(11, 4))
# A slope plot, not a boxplot. Every grey line is one seed carried across the
# conditions, so the pairing the whole analysis relies on is visible: what matters is
# whether the lines tilt the same way, not how wide the spread is.
for seed in SEEDS:
    axes[0].plot(range(len(CONDITIONS)), [values[c][seed] for c in CONDITIONS],
                 marker="o", ms=3, lw=0.8, alpha=0.45, color="#6b7280")
axes[0].plot(range(len(CONDITIONS)), [means[c] for c in CONDITIONS],
             marker="o", ms=8, lw=2.5, color="#A31F34", label="mean")
axes[0].set_xticks(range(len(CONDITIONS)))
axes[0].set_xticklabels([c.replace("_", "-") for c in CONDITIONS])
axes[0].set_ylabel("held-out bits/dim")
axes[0].set_title("each seed, across the three orders")
axes[0].legend(fontsize=8)

names = [row["name"] for row in report["rows"]]
axes[1].barh(names, [row["mean"] for row in report["rows"]], color="#A31F34")
for index, row in enumerate(report["rows"]):
    axes[1].plot([-row["floor"], row["floor"]], [index, index], color="#111827", lw=2)
axes[1].axvline(0, color="#6b7280", lw=1)
# Drawn on both sides, so the band is right whichever direction you declared.
axes[1].axvspan(-sesoi_absolute, sesoi_absolute, color="#1D4ED8", alpha=0.12)
axes[1].set_xlabel("paired difference in bits/dim (bar), floor (black), inside blue is\n"
                   "below your SESOI")
axes[1].set_title("is the difference bigger than the noise?")
plt.tight_layout()
plt.show()

## Your task 6: Write the record

The record has six fields. `claim` and `evidence` say what you found and the numbers
behind it. `resolved_comparisons` says which comparisons cleared their floor.
`caveat` says what your design could not control. `not_supported` says what a reader
might reasonably conclude from your result that it does not actually show.
`next_experiment` names the single change you would make next.

Three fields are checked for form, though not for quality. `caveat` has to name
something the design held constant, because that marks the edge of what your result
covers. `resolved_comparisons` has to use the verdicts the analysis returned, so write
it from the output. `not_supported` has to say something your claim does not.

Before you write, look at the line the results cell printed comparing your
`FIRST_CALL` with what the run found. If the run disagreed with you, say so in
`claim`.

If your order won, it does not follow that you know why it won, and this experiment
measured only likelihood, so it says nothing about sample quality. If your order made
no resolvable difference, report that as the result, stated against your declared SESOI
and this model's capacity.

In [ ]:
# STUDENT TASK 6: the experiment record. Every field must be non-empty.
learner_record = {
    "claim": "",             # TODO: what you found, in one sentence, with the numbers.
    "evidence": "",          # TODO: the measurements that support the claim.
    "resolved_comparisons": "",  # TODO: which contrasts cleared their floor, and by how much.
    "caveat": "",            # TODO: what your design could not control.
    "not_supported": "",     # TODO: what your result does NOT show.
    "next_experiment": "",   # TODO: the single change you would make next.
}

## Checking what you ran

The checks below are the same ones a research group would run before believing its
own result. Three of them are specific to this project and two of those decide
whether the comparison means anything at all.

Connectivity must be bit-identical across conditions, or you measured capacity rather
than order. Every order must be a genuine permutation, or the likelihoods are not
comparable because they describe different densities. And the network is checked for
future leakage by gradient rather than by trusting the masks: the logit at rank $r$
must have exactly zero gradient with respect to every input at rank $r$ or later.

In [ ]:
result = adapter.run_all_checks(
    plan=plan, records=records, configs=configs, orders=orders,
    signatures=signatures, eval_hashes=eval_hashes, model=last_model.cpu(),
    dims=DIMS, record=learner_record, budget_min=BUDGET_MIN, budget_max=BUDGET_MAX,
    metric=METRIC, provenance=provenance,
    measured_budget=measured_budget_units)

execution_contract, execution_report = schema.checklist({
    "every contract check passed, listed above": bool(result.passed),
    "your own Task 2 and Task 3 helpers reproduce the reported analysis":
        helpers_agree,
    "connectivity is bit-identical across every order":
        len(set(signatures.values())) == 1,
    "your projected budget is above zero, so the comparison below means "
    "something": projected_budget_units > 0,
    "the compute you spent matches what you projected, within 10%":
        projected_budget_units > 0
        and abs(measured_budget_units - projected_budget_units)
        <= 0.10 * projected_budget_units,
})

print(f"contract passed: {result.passed}")
print(result.report())
print(f"\nR6 execution contract: {execution_contract}")
print(execution_report)

## Report values

Run the cell below once every task is complete and the experiment has finished. It
prints seven labelled values, R1 through R7. Copy each into the box with the matching
label on the course page.

R1 to R3 are your three analysis helpers run on the inputs Checkpoint 1 issued you.
R4 and R5 describe your design before the run. R6 and R7 describe what the run
actually did. The last two checkpoints are answered on the course page and do not use
values from this notebook.

In [ ]:
probe_budget = probe_budget_value if budget_probe_contract else -1.0
probe_paired = probe_paired_value if paired_probe_contract else -1.0
probe_floor = probe_floor_value if floor_probe_contract else -1.0

report_values = {
    "R1: compute budget on your issued inputs": issued_budget_value,
    "R2: paired difference on your issued table": issued_paired_value,
    "R3: resolution floor on your issued table": issued_floor_value,
    "R4: design contract": design_contract,
    "R5: projected budget units for your design": projected_budget_units,
    "R6: execution contract": execution_contract,
    "R7: measured budget units your experiment spent": measured_budget_units,
}
# Each line below names the task that produces it, so a failure points to what to fix.
ready, ready_report = schema.checklist({
    "issued inputs copied from Checkpoint 1 (block above Task 1)": issued_ready,
    "R1 matches its fixed probe (Task 1, compute_budget)":
        abs(probe_budget - 0.4375) < 1e-4,
    "R2 matches its fixed probe (Task 2, paired_difference)":
        abs(probe_paired - (-0.04)) < 1e-4,
    "R3 matches its fixed probe (Task 3, resolution_floor)":
        abs(probe_floor - 0.0248) < 1e-4,
    "R4 design contract passed (Tasks 4 and 5)": design_contract == 1,
    "R6 execution contract passed (the run and Task 6)": execution_contract == 1,
})
if not ready:
    print("Not ready to submit. Nothing is printed below until these pass:")
    print(ready_report)
else:
    print("CAPSTONE REPORT VALUES")
    for label, value in report_values.items():
        print(f"{label}: {value}")